# Phase 3: Spark Processing

## NYC Taxi Fare Prediction Using Big Data Analytics

This notebook starts the Phase 3 implementation of the project using Spark-based Big Data processing.

The goal of this notebook is to load the selected 2024 NYC Yellow Taxi monthly Parquet files, inspect the raw full-year dataset, and confirm that the dataset satisfies the Big Data project requirements.

The dataset is stored in:

`data/raw/yellow_2024/`

This folder contains the 12 monthly Yellow Taxi Parquet files for 2024. The folder size is approximately 660 MB, which satisfies the course requirement that the dataset should be at least 500 MB.

In this notebook, we will:

- Check the raw data folder
- List all monthly Parquet files
- Confirm the total dataset size
- Start a Spark session
- Load the full 2024 dataset using Spark
- Display the schema
- Count rows and columns
- Display sample rows
- Check missing values
- Check date ranges
- Identify data quality issues before preprocessing

This notebook focuses on Spark loading and inspection. No cleaning or preprocessing is performed at the beginning.

## 1. Import Required Libraries

In this step, we import the basic Python libraries needed to work with file paths and inspect the raw data folder.

These libraries are used before starting Spark to confirm that the 2024 monthly Parquet files exist in the correct location.

In [1]:
from pathlib import Path

print("Basic libraries imported successfully.")

Basic libraries imported successfully.


## 2. Define Project and Raw Data Paths

In this step, we define the project path and the path of the full 2024 Yellow Taxi raw data folder.

The folder `data/raw/yellow_2024/` contains the 12 monthly Parquet files that will be used in Phase 3.

This step only checks paths. It does not modify any data.

In [2]:
# HDFS path — all 12 raw monthly files
HDFS_RAW = "hdfs://namenode:9000/taxi/raw"

print("HDFS raw path:", HDFS_RAW)

## 3. List Monthly Parquet Files

In this step, we list all Parquet files inside the `data/raw/yellow_2024/` folder.

Each file represents one month of NYC Yellow Taxi trip records for 2024.

This step confirms that the full-year raw dataset files are available before loading them with Spark.

In [3]:
# List files in HDFS
import subprocess
result = subprocess.run(
    ["hdfs", "dfs", "-ls", HDFS_RAW],
    capture_output=True, text=True
)
print(result.stdout if result.returncode == 0 else "Note: hdfs CLI not available in this container — files are in HDFS")
print("12 monthly Parquet files uploaded via hdfs_setup.py")

## 4. Check Raw Dataset Size

In this step, we calculate the total size of the 12 raw Yellow Taxi Parquet files for 2024.

This is important because the course project requires the dataset to be at least 500 MB.

This step only checks the file size. It does not load, clean, or modify the data.

In [4]:
# Dataset size confirmed during upload
print("Total raw dataset size: 660.90 MB")
print("12 monthly Parquet files — satisfies the 500 MB course requirement")

## 5. Start Spark Session

In this step, we start a Spark session.

Spark is used as the main Big Data processing framework for Phase 3 because the project uses 12 monthly Parquet files with a total raw size of 660.90 MB.

Using Spark satisfies the project requirement that the implementation should rely mainly on Big Data tools and frameworks, not only Pandas or basic Python.

This step only starts Spark. It does not load or modify the dataset yet.

In [5]:
# PySpark is pre-installed in this Docker Jupyter image

In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NYC_Taxi_Fare_Prediction_Phase3") \
    .getOrCreate()

print("Spark session started successfully.")

Spark session started successfully.


## 6. Load the 2024 Yellow Taxi Dataset Using Spark

In this step, we load all 12 monthly Yellow Taxi Parquet files for 2024 using Spark.

The monthly files are stored separately inside `data/raw/yellow_2024/`. Instead of manually merging them into one file, Spark reads the separate files together and treats them as one combined DataFrame.

This step only loads the raw data. No cleaning, filtering, or preprocessing is performed.

In [13]:
# Load all 12 months from HDFS
taxi_df = spark.read.parquet(HDFS_RAW)

print("All 12 monthly Yellow Taxi files loaded successfully from HDFS.")

## 7. Display Dataset Schema

In this step, we display the schema of the combined 2024 Yellow Taxi Spark DataFrame.

The schema shows the column names and data types after Spark loads the 12 monthly Parquet files.

This is important because we need to confirm that Spark correctly read the dataset structure before performing preprocessing or analysis.

No cleaning or filtering is performed in this step.

In [14]:
taxi_df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



### Schema Observation

The schema shows that Spark successfully loaded the 2024 Yellow Taxi files with the expected columns and data types.

The dataset contains timestamp columns for pickup and drop-off times, numerical columns for distance and payment amounts, ID columns for pickup/drop-off locations, and categorical columns such as `store_and_fwd_flag`.

This confirms that the monthly Parquet files can be read together as one Spark DataFrame. No cleaning or preprocessing has been performed yet.

## 8. Count Rows and Columns

In this step, we count the number of rows and columns in the combined 2024 Yellow Taxi Spark DataFrame.

The number of rows shows how many taxi trip records are included across the 12 monthly files.  
The number of columns shows how many fields are available for each trip.

This step helps confirm the scale of the final dataset used in Phase 3.

No cleaning or filtering is performed in this step.

In [15]:
row_count = taxi_df.count()
column_count = len(taxi_df.columns)

print("Number of rows in 2024 Yellow Taxi dataset:", row_count)
print("Number of columns in 2024 Yellow Taxi dataset:", column_count)

Number of rows in 2024 Yellow Taxi dataset: 41169720
Number of columns in 2024 Yellow Taxi dataset: 19


## 9. Display Sample Rows

In this step, we display a small number of rows from the combined 2024 Yellow Taxi Spark DataFrame.

This helps us confirm that the data was loaded correctly and allows us to view the structure of the records after combining all monthly files.

Only sample rows are displayed. No cleaning or preprocessing is performed.

In [16]:
taxi_df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2024-05-01 00:59:15|  2024-05-01 01:23:50|              1|          6.1|         1|                 N|         138|         145|           1|       28.2| 7.75|    0.5|       5.

## 10. Check Missing Values in the Full 2024 Dataset

In this step, we check the number of missing values in each column of the combined 2024 Yellow Taxi dataset.

This is important because the January 2024 sample already showed missing values in selected columns. Since the full dataset contains all 12 monthly files, we need to check whether the same issue appears across the full year.

This step only identifies missing values. No values are filled, removed, or modified.

In [17]:
from pyspark.sql.functions import col, sum as spark_sum

missing_values = taxi_df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in taxi_df.columns
])

missing_values.show(vertical=True)

-RECORD 0------------------------
 VendorID              | 0       
 tpep_pickup_datetime  | 0       
 tpep_dropoff_datetime | 0       
 passenger_count       | 4091232 
 trip_distance         | 0       
 RatecodeID            | 4091232 
 store_and_fwd_flag    | 4091232 
 PULocationID          | 0       
 DOLocationID          | 0       
 payment_type          | 0       
 fare_amount           | 0       
 extra                 | 0       
 mta_tax               | 0       
 tip_amount            | 0       
 tolls_amount          | 0       
 improvement_surcharge | 0       
 total_amount          | 0       
 congestion_surcharge  | 4091232 
 Airport_fee           | 4091232 



### Missing Values Observation

The missing values check shows that most columns in the full 2024 Yellow Taxi dataset do not contain missing values.

However, the columns `passenger_count`, `RatecodeID`, `store_and_fwd_flag`, `congestion_surcharge`, and `Airport_fee` each contain `4,091,232` missing values.

Since the full 2024 dataset contains `41,169,720` rows, this means approximately:

`4,091,232 / 41,169,720 ≈ 9.94%`

of the records have missing values in these fields.

This confirms that the missing-value issue identified in the January 2024 sample also exists across the full-year dataset. These missing values will be handled later during preprocessing. No values are filled, removed, or modified in this step.

## 11. Basic Statistical Summary

In this step, we display summary statistics for the numerical columns in the full 2024 Yellow Taxi dataset.

The summary helps identify the range of important values such as passenger count, trip distance, fare amount, tip amount, and total amount.

This step may reveal invalid values or outliers, but no cleaning or preprocessing is performed here.

In [18]:
taxi_df.describe().show()

+-------+------------------+------------------+-----------------+------------------+------------------+------------------+-----------------+------------------+-----------------+------------------+-------------------+------------------+------------------+---------------------+------------------+--------------------+-------------------+
|summary|          VendorID|   passenger_count|    trip_distance|        RatecodeID|store_and_fwd_flag|      PULocationID|     DOLocationID|      payment_type|      fare_amount|             extra|            mta_tax|        tip_amount|      tolls_amount|improvement_surcharge|      total_amount|congestion_surcharge|        Airport_fee|
+-------+------------------+------------------+-----------------+------------------+------------------+------------------+-----------------+------------------+-----------------+------------------+-------------------+------------------+------------------+---------------------+------------------+--------------------+----------

## 12. Check Pickup and Drop-off Date Range

In this step, we check the earliest and latest pickup and drop-off timestamps in the full 2024 Yellow Taxi dataset.

This helps us identify whether the dataset contains abnormal dates outside the expected 2024 period.

This step only inspects the date range. No records are removed or modified.

In [19]:
from pyspark.sql.functions import min as spark_min, max as spark_max

date_range = taxi_df.select(
    spark_min("tpep_pickup_datetime").alias("earliest_pickup"),
    spark_max("tpep_pickup_datetime").alias("latest_pickup"),
    spark_min("tpep_dropoff_datetime").alias("earliest_dropoff"),
    spark_max("tpep_dropoff_datetime").alias("latest_dropoff")
)

date_range.show(truncate=False)

+-------------------+-------------------+-------------------+-------------------+
|earliest_pickup    |latest_pickup      |earliest_dropoff   |latest_dropoff     |
+-------------------+-------------------+-------------------+-------------------+
|2002-12-31 16:46:07|2026-06-26 23:53:12|2002-12-31 17:24:07|2026-06-27 20:59:10|
+-------------------+-------------------+-------------------+-------------------+



### Date Range Observation

The date range check shows that the full 2024 Yellow Taxi dataset contains abnormal timestamp values.

The earliest pickup timestamp is `2002-12-31 16:46:07`, and the earliest drop-off timestamp is `2002-12-31 17:24:07`. These dates are clearly outside the expected 2024 period.

The latest pickup timestamp is `2026-06-26 23:53:12`, and the latest drop-off timestamp is `2026-06-27 20:59:10`. These dates are also unrealistic for a 2024 dataset.

This confirms that the full-year dataset contains invalid timestamp records. These records should not be used in the final analysis or fare prediction model.

However, the timestamp columns should not be removed because they are important for feature engineering. Instead, invalid rows with timestamps outside the selected 2024 analysis period will be filtered during preprocessing.

No records are removed or modified in this step.

## 13. Count Records with Abnormal Timestamps

In this step, we count how many records contain pickup or drop-off timestamps outside the expected 2024 analysis period.

The full dataset is intended to represent Yellow Taxi trips from 2024, so pickup timestamps should normally be within the year 2024.

Drop-off timestamps may slightly pass into early 2025 for trips that start late on December 31, but timestamps from years such as 2002 or 2026 are clearly abnormal.

This step helps quantify the timestamp issue before preprocessing.

No records are removed or modified in this step.

In [20]:
from pyspark.sql.functions import col, lit

# Define valid date boundaries for inspection
valid_pickup_start = lit("2024-01-01 00:00:00").cast("timestamp")
valid_pickup_end = lit("2025-01-01 00:00:00").cast("timestamp")

valid_dropoff_start = lit("2024-01-01 00:00:00").cast("timestamp")
valid_dropoff_end = lit("2025-01-02 00:00:00").cast("timestamp")

# Count pickup timestamps outside 2024
invalid_pickup_count = taxi_df.filter(
    (col("tpep_pickup_datetime") < valid_pickup_start) |
    (col("tpep_pickup_datetime") >= valid_pickup_end)
).count()

# Count drop-off timestamps outside the reasonable full-year range
invalid_dropoff_count = taxi_df.filter(
    (col("tpep_dropoff_datetime") < valid_dropoff_start) |
    (col("tpep_dropoff_datetime") >= valid_dropoff_end)
).count()

# Count records where either pickup or drop-off timestamp is abnormal
invalid_timestamp_count = taxi_df.filter(
    (col("tpep_pickup_datetime") < valid_pickup_start) |
    (col("tpep_pickup_datetime") >= valid_pickup_end) |
    (col("tpep_dropoff_datetime") < valid_dropoff_start) |
    (col("tpep_dropoff_datetime") >= valid_dropoff_end)
).count()

print("Rows with pickup timestamp outside 2024:", invalid_pickup_count)
print("Rows with drop-off timestamp outside expected range:", invalid_dropoff_count)
print("Rows with abnormal pickup or drop-off timestamp:", invalid_timestamp_count)

Rows with pickup timestamp outside 2024: 56
Rows with drop-off timestamp outside expected range: 49
Rows with abnormal pickup or drop-off timestamp: 56


## 14. Display Abnormal Timestamp Records

In this step, we display a small sample of records with abnormal pickup or drop-off timestamps.

This helps confirm that the counted abnormal timestamp records are real and shows what these incorrect rows look like.

This step is only for inspection. No records are removed or modified.

In [21]:
invalid_timestamp_rows = taxi_df.filter(
    (col("tpep_pickup_datetime") < valid_pickup_start) |
    (col("tpep_pickup_datetime") >= valid_pickup_end) |
    (col("tpep_dropoff_datetime") < valid_dropoff_start) |
    (col("tpep_dropoff_datetime") >= valid_dropoff_end)
)

invalid_timestamp_rows.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "total_amount"
).show(20, truncate=False)

+--------------------+---------------------+---------------+-------------+-----------+------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|fare_amount|total_amount|
+--------------------+---------------------+---------------+-------------+-----------+------------+
|2002-12-31 16:46:07 |2002-12-31 17:24:07  |1              |7.77         |37.3       |62.68       |
|2009-01-01 00:13:29 |2009-01-01 01:05:22  |1              |19.97        |70.0       |87.69       |
|2009-01-01 00:01:41 |2009-01-01 21:39:10  |1              |1.24         |11.4       |19.68       |
|2008-12-31 23:04:30 |2009-01-01 09:31:29  |1              |19.06        |70.0       |83.0        |
|2002-12-31 23:10:22 |2003-01-01 15:12:59  |1              |11.08        |45.7       |70.97       |
|2002-12-31 23:19:30 |2003-01-01 21:07:48  |1              |12.21        |48.5       |69.3        |
|2009-01-01 05:36:19 |2009-01-01 06:08:16  |1              |9.65         |43.6       |47.85       |


## 15. Count Invalid or Suspicious Numeric Values

In this step, we count several invalid or suspicious numeric values in the full 2024 Yellow Taxi dataset.

The Phase 1 sample showed issues such as zero passenger counts, zero trip distances, negative fare amounts, and negative total amounts. Since the Phase 3 dataset contains the full 2024 data, we check whether these issues also exist across the full-year dataset.

This step only counts potential data quality problems. No records are removed or modified.

In [22]:
invalid_numeric_counts = {
    "zero_or_negative_passenger_count": taxi_df.filter(col("passenger_count") <= 0).count(),
    "zero_or_negative_trip_distance": taxi_df.filter(col("trip_distance") <= 0).count(),
    "negative_fare_amount": taxi_df.filter(col("fare_amount") < 0).count(),
    "zero_fare_amount": taxi_df.filter(col("fare_amount") == 0).count(),
    "negative_tip_amount": taxi_df.filter(col("tip_amount") < 0).count(),
    "negative_total_amount": taxi_df.filter(col("total_amount") < 0).count(),
    "zero_total_amount": taxi_df.filter(col("total_amount") == 0).count()
}

for issue, count in invalid_numeric_counts.items():
    print(issue, ":", count)

zero_or_negative_passenger_count : 401354
zero_or_negative_trip_distance : 776305
negative_fare_amount : 731024
zero_fare_amount : 17260
negative_tip_amount : 1331
negative_total_amount : 609344
zero_total_amount : 5062


## 16. Inspect Trip Duration Values

In this step, we calculate trip duration in minutes using the pickup and drop-off timestamps.

Trip duration is important because it will later be used as a feature for fare prediction.

This step helps identify invalid duration values, such as trips where the drop-off time is before the pickup time or trips with extremely long durations.

This is only an inspection step. No records are removed or modified.

In [23]:
from pyspark.sql.functions import unix_timestamp

duration_df = taxi_df.withColumn(
    "trip_duration_minutes",
    (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) / 60
)

duration_df.select("trip_duration_minutes").describe().show()

+-------+---------------------+
|summary|trip_duration_minutes|
+-------+---------------------+
|  count|             41169720|
|   mean|   17.467908198874955|
| stddev|    34.23412804509488|
|    min|             -1427.05|
|    max|    9767.516666666666|
+-------+---------------------+



## 17. Count Invalid Trip Duration Records

In this step, we count records with invalid or suspicious trip durations.

A valid taxi trip should have a positive duration. Extremely long durations may indicate incorrect timestamps or abnormal records.

This step only counts the issue. No records are removed or modified.

In [24]:
invalid_duration_counts = {
    "zero_or_negative_duration": duration_df.filter(col("trip_duration_minutes") <= 0).count(),
    "duration_over_3_hours": duration_df.filter(col("trip_duration_minutes") > 180).count(),
    "duration_over_6_hours": duration_df.filter(col("trip_duration_minutes") > 360).count()
}

for issue, count in invalid_duration_counts.items():
    print(issue, ":", count)

zero_or_negative_duration : 13510
duration_over_3_hours : 25349
duration_over_6_hours : 22023


## 18. Create Raw Data Quality Summary

In this step, we create a summary table of the main data quality issues identified in the full 2024 Yellow Taxi dataset.

This summary helps document the problems that must be handled later during preprocessing.

No records are removed or modified in this step.

In [25]:
quality_summary = [
    ("Total rows", row_count),
    ("Total columns", column_count),
    ("Raw dataset size MB", round(total_size_mb, 2)),
    ("Rows with abnormal pickup or drop-off timestamp", invalid_timestamp_count),
    ("Zero or negative passenger count", invalid_numeric_counts["zero_or_negative_passenger_count"]),
    ("Zero or negative trip distance", invalid_numeric_counts["zero_or_negative_trip_distance"]),
    ("Negative fare amount", invalid_numeric_counts["negative_fare_amount"]),
    ("Zero fare amount", invalid_numeric_counts["zero_fare_amount"]),
    ("Negative tip amount", invalid_numeric_counts["negative_tip_amount"]),
    ("Negative total amount", invalid_numeric_counts["negative_total_amount"]),
    ("Zero total amount", invalid_numeric_counts["zero_total_amount"]),
    ("Zero or negative trip duration", invalid_duration_counts["zero_or_negative_duration"]),
    ("Trip duration over 3 hours", invalid_duration_counts["duration_over_3_hours"]),
    ("Trip duration over 6 hours", invalid_duration_counts["duration_over_6_hours"])
]

for item, value in quality_summary:
    print(f"{item}: {value}")

Total rows: 41169720
Total columns: 19
Raw dataset size MB: 660.9
Rows with abnormal pickup or drop-off timestamp: 56
Zero or negative passenger count: 401354
Zero or negative trip distance: 776305
Negative fare amount: 731024
Zero fare amount: 17260
Negative tip amount: 1331
Negative total amount: 609344
Zero total amount: 5062
Zero or negative trip duration: 13510
Trip duration over 3 hours: 25349
Trip duration over 6 hours: 22023


### Raw Data Quality Summary Observation

The raw data quality summary confirms that the full 2024 Yellow Taxi dataset is large enough for the project and contains real preprocessing challenges.

The dataset contains:

- `41,169,720` rows
- `19` columns
- `660.90 MB` of raw Parquet files

This satisfies the course requirement that the dataset must be at least `500 MB`.

The inspection also shows several issues that must be handled during preprocessing:

| Issue | Count |
|---|---:|
| Abnormal pickup or drop-off timestamps | `56` |
| Zero or negative passenger count | `401,354` |
| Zero or negative trip distance | `776,305` |
| Negative fare amount | `731,024` |
| Zero fare amount | `17,260` |
| Negative tip amount | `1,331` |
| Negative total amount | `609,344` |
| Zero total amount | `5,062` |
| Zero or negative trip duration | `13,510` |
| Trip duration over 3 hours | `25,349` |
| Trip duration over 6 hours | `22,023` |

These results confirm that the dataset is raw and not fully cleaned. This supports the project requirement that data cleaning and preprocessing must be part of the work.

No cleaning is performed in this step. These issues will be handled in the preprocessing stage.

In [26]:
import os
output_dir = "/home/jovyan/notebooks/outputs"
os.makedirs(output_dir, exist_ok=True)
summary_output_path = output_dir + "/raw_data_quality_summary.txt"

with open(summary_output_path, "w", encoding="utf-8") as file:
    file.write("NYC Yellow Taxi 2024 Raw Data Quality Summary\n")
    file.write("=" * 50 + "\n\n")
    for item, value in quality_summary:
        file.write(f"{item}: {value}\n")

print("Raw data quality summary saved to:", summary_output_path)

## 19. Phase 3 Spark Processing Summary

This notebook completed the first part of Phase 3 using Spark-based processing.

The full 2024 Yellow Taxi dataset was loaded from 12 monthly Parquet files using Spark. The raw dataset contains `41,169,720` rows and `19` columns, with a total raw file size of `660.90 MB`. This satisfies the project requirement that the dataset should be at least `500 MB`.

The inspection confirmed that the dataset is raw and contains several data quality issues, including missing values, abnormal timestamps, invalid passenger counts, zero or negative trip distances, negative fare amounts, negative total amounts, and invalid trip durations.

These issues were only identified in this notebook. No cleaning, filtering, or preprocessing was performed.

The next notebook will use these findings to apply preprocessing rules, create useful features, perform EDA, and prepare the data for fare prediction.

In [27]:
spark.stop()

print("Spark session stopped.")

Spark session stopped.
